# RamanBench — Reproducing the Full Benchmark

This notebook documents how the paper benchmark results were produced and how to reproduce them.

## What the benchmark measures

| Dimension | Count |
|---|---|
| Models | 29 (traditional ML, deep learning, tabular foundation, Raman-specific, AutoML) |
| Datasets | 77 (56 regression, 21 classification) |
| Seeds | 3 |
| **Total (model, dataset, seed) triples** | **~6 700** |

## Ecosystem

| Resource | Link |
|---|---|
| **raman-data** (datasets) | [GitHub](https://github.com/ml-lab-htw/raman_data) · [PyPI](https://pypi.org/project/raman-data/) |
| **raman-bench** (this package) | [GitHub](https://github.com/ml-lab-htw/RamanBench) · [PyPI](https://pypi.org/project/raman-bench/) |
| **AutoGluon fork** | [GitHub](https://github.com/mario-koddenbrock/autogluon) |
| **Live Leaderboard** | [HuggingFace Space](https://huggingface.co/spaces/ml-lab-htw/RamanBench) |
| **Paper** | [arXiv TBD](https://arxiv.org/abs/TBD) |

## 1 — Prerequisites

The full benchmark requires the **AutoGluon fork**, which removes the 500-feature cap on
tabular foundation models and upgrades TabICL to v2 (regression support).  Standard
`pip install autogluon` will not work for reproducing the paper results.

Install order matters — the fork must be installed **before** `raman-bench`:

In [1]:
# Option A — install fork from GitHub (one-liner, takes ~10 min)
# !pip install -r https://raw.githubusercontent.com/ml-lab-htw/RamanBench/main/requirements-autogluon-fork.txt

# Option B — install from a local clone of the fork
# FORK=/path/to/autogluon_fork
# !pip install -e $FORK/common $FORK/core $FORK/features $FORK/tabular

# Then install the benchmark packages
# !pip install -e /path/to/RamanBench[full]
# !pip install raman-data

Verify the installation:

In [2]:
from raman_bench.model import AutoGluonModel
from raman_bench.preprocessing.wrapped_models import Prep_PLS
from raman_bench.predictions import compute_predictions
from raman_bench.config import load_config

import autogluon.tabular
print("AutoGluon version:", autogluon.tabular.__version__)

# Confirm TabICL v2 is available (fork-only)
try:
    from autogluon.tabular.models import TabICLModel
    print("TabICL v2: available")
except ImportError:
    print("TabICL v2: NOT available — check fork install")

ModuleNotFoundError: No module named 'torch'

## 2 — Benchmark configuration

In [ ]:
import json
from raman_bench.config import load_config
from raman_bench.benchmark import configure_benchmark

cfg = load_config("configs/v1_default.json")
bm  = configure_benchmark(cfg)

print(f"Models:   {len(cfg['models'])} — {', '.join(cfg['models'][:6])}, ...")
print(f"Datasets: {len(bm)} total ({bm.n_regression} regression, {bm.n_classification} classification)")
print(f"Seeds:    {cfg['n_repetitions']}")
print(f"Triples:  {len(cfg['models']) * len(bm) * cfg['n_repetitions']}")
print(f"Output:   {cfg['output_dir']}")

## 3 — Local test run (single model, debug config)

Before running on the cluster, verify the pipeline end-to-end on a small subset.
The `debug.json` config runs 2 datasets × 1 seed.

In [ ]:
import subprocess, sys

result = subprocess.run(
    [sys.executable, "scripts/run_benchmark.py",
     "--config", "configs/debug.json",
     "--model",  "PLS",
     "--step",   "predictions"],
    capture_output=True, text=True
)
print(result.stdout[-3000:] if len(result.stdout) > 3000 else result.stdout)
if result.returncode != 0:
    print("STDERR:", result.stderr[-1000:])

In [ ]:
# Inspect what was written
import pandas as pd
from pathlib import Path

debug_cfg = json.load(open("configs/debug.json"))
pred_dir  = Path(debug_cfg["output_dir"]) / "seed_0" / "predictions"

files = sorted(pred_dir.glob("*.csv"))
print(f"{len(files)} prediction file(s) written to {pred_dir}")
if files:
    df = pd.read_csv(files[0])
    print(f"\nExample — {files[0].name}:")
    print(df.head())

## 4 — Full benchmark on a SLURM cluster

The full run is a single SLURM job that trains all 29 models across all datasets and seeds
sequentially.  Prediction files are checkpointed as each (model, dataset, seed) triple
finishes, so the job can be restarted without losing work.

**Estimated wall time:** 3–5 days on a single GPU node.

### One-time cluster setup

```bash
BASE=/home/cluster_home/koddenb/workspace

# Create conda environment
conda create -n autogluon python=3.11 -y
conda activate autogluon

# Clone repos (AutoGluon fork already at $BASE/autogluon_fork)
git clone https://github.com/ml-lab-htw/RamanBench.git        $BASE/RamanBench
git clone https://github.com/mario-koddenbrock/raman_bench_paper.git  $BASE/raman_bench_paper

# Install AutoGluon fork first
pip install -e $BASE/autogluon_fork/common
pip install -e $BASE/autogluon_fork/core
pip install -e $BASE/autogluon_fork/features
pip install -e $BASE/autogluon_fork/tabular

# Install benchmark packages
pip install raman-data
pip install -e "$BASE/RamanBench[full]"
pip install -e "$BASE/raman_bench_paper"

# Create required directories
mkdir -p $BASE/raman_bench_paper/.logs
mkdir -p /scratch/koddenbrock/raman_bench/.cache/autogluon
```

### Submit

```bash
cd $BASE/raman_bench_paper

# Dry run first — prints what would be submitted without submitting
bash cluster/submit_v1_full.sh --dry-run

# Submit the default condition
bash cluster/submit_v1_full.sh --default-only
```

### Monitor

```bash
squeue -u $USER
tail -f .logs/<job_id>_RB_v1_default.log

# Count completed prediction files
find results/v1_default -name '*.csv' | wc -l
```

## 5 — After the run: metrics and leaderboard

Once predictions are done, compute metrics.  This step runs locally in seconds.

In [ ]:
import subprocess, sys

result = subprocess.run(
    [sys.executable, "scripts/run_benchmark.py",
     "--config", "configs/v1_default.json",
     "--step",   "metrics"],
    capture_output=True, text=True
)
print(result.stdout)
if result.returncode != 0:
    print("STDERR:", result.stderr[-500:])

In [ ]:
import pandas as pd
from pathlib import Path
from raman_bench import Leaderboard

metrics_dir = Path("results/v1_default/metrics")

lb = Leaderboard(
    reg_metrics=pd.read_csv(metrics_dir / "regression_metrics.csv"),
    clf_metrics=pd.read_csv(metrics_dir / "classification_metrics.csv"),
)
lb.rank().head(15)

In [ ]:
import matplotlib.pyplot as plt

fig = lb.plot(task="overall", n_top=29)
plt.tight_layout()
plt.show()

## 6 — Precomputed results (no cluster needed)

The benchmark results are shipped with the package as precomputed baselines.
This lets anyone explore the leaderboard and add new models without running the cluster.
See **notebook 02** for a full walkthrough of adding new models.

In [ ]:
from raman_bench import Leaderboard

lb = Leaderboard.from_precomputed()
print("Loaded precomputed leaderboard")
lb.rank().head(10)